# Part D: frozen AraBERT

`aubmindlab/bert-base-arabertv02` is never trained here. It turns each sentence into vectors once, and only a small model learns on top. The text goes through the AraBERT preprocessor, as its model card asks (Q3).

1. Extract the BERT states
2. D1: [CLS] against mean of the tokens, with logistic regression
3. D2: all the token states, with the BiLSTM + attention head of C2
4. Attention on 5 examples
5. Full pipeline on val


**Running this notebook.** Each model has its own cell, and a finished run is saved on Drive. If a cell fails, rerun only that cell. After a lost session, run the notebook again: finished runs reload in seconds. To train one again, pass `retrain=True` in its cell.

## Setup

In [ ]:
import os
import sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "AR-aspect-sentiment"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q arabert emoji gensim sentencepiece tashaphyne

OUTPUT_DIR = f"/content/drive/MyDrive/{REPO}"
os.environ["OUTPUT_DIR"] = OUTPUT_DIR
!mkdir -p {OUTPUT_DIR}

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import seaborn as sns
import torch
from arabert.preprocess import ArabertPreprocessor
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

from src.data import POLARITIES, category_list, category_matrix, load_test, load_train_val, pair_units, pairs
from src.evaluation import (
    THRESHOLD_GRID,
    best_category_thresholds,
    best_global_threshold,
    category_report,
    category_scores,
    compare_threshold_policies,
    pairs_from_matrix,
    pipeline_outcomes,
    pipeline_scores,
    polarity_scores,
    predict_labels,
)
from src.plots import (
    plot_category_f1,
    plot_f1_by_threshold,
    plot_model_comparison,
    plot_pipeline_outcomes,
    plot_threshold_policies,
    plot_training_curves,
    show_attention,
)
from src.tracking import best_of_each_model, cached_run, latency_ms, log_results, result_row, save_scores
from src.training import DEVICE, collect_outputs, compact_state, fit, set_seed

SEED = 42
BERT_NAME = "aubmindlab/bert-base-arabertv02"
MAX_LENGTH = 128
BATCH_SIZE = 32
HEAD_HIDDEN_SIZE = 128
DROPOUT = 0.3
MAX_EPOCHS = 40
PATIENCE = 6
PREPROCESSING = "arabert"
sns.set_theme(style="whitegrid", context="notebook")
print("Device:", DEVICE)

In [ ]:
opinions, train, val = load_train_val(SEED)

classes = category_list(opinions)
category_ids = {name: index for index, name in enumerate(classes)}

train_labels = category_matrix(train.categories, classes)
val_labels = category_matrix(val.categories, classes)

val_pairs = pairs(opinions, val)
train_units = pair_units(pairs(opinions, train))
train_units = train_units[~train_units.ambiguous].reset_index(drop=True)
val_units = pair_units(val_pairs)

print(f"{len(train)} train and {len(val)} val sentences, {len(train_units)} and {len(val_units)} pairs")

## 1. Extract the BERT states

Each sentence is read once. Three things are kept: the [CLS] state, the mean of the token states, and all the token states for D2.

In [ ]:
preprocess_text = ArabertPreprocessor(BERT_NAME).preprocess
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
bert = AutoModel.from_pretrained(BERT_NAME).to(DEVICE).eval()
BERT_SIZE = bert.config.hidden_size


def bert_states(texts, batch_size=64):
    start_time = time.time()
    cls_vectors, mean_vectors, token_states = [], [], []
    prepared_texts = [preprocess_text(text) for text in texts]
    for start in tqdm(range(0, len(prepared_texts), batch_size), desc="BERT states", leave=False):
        batch = tokenizer(prepared_texts[start:start + batch_size], truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            states = bert(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(2)
        cls_vectors.append(states[:, 0].cpu().numpy())
        mean_vectors.append(((states * mask).sum(dim=1) / mask.sum(dim=1)).cpu().numpy())
        token_states += [row[:length].half().cpu() for row, length in zip(states, batch["attention_mask"].sum(dim=1))]
    return {"cls": np.concatenate(cls_vectors), "mean": np.concatenate(mean_vectors), "tokens": token_states, "seconds": time.time() - start_time}

In [ ]:
train_features = cached_run("v2_arabert_train_states", lambda: bert_states(train.text.tolist()))

In [ ]:
val_features = cached_run("v2_arabert_val_states", lambda: bert_states(val.text.tolist()))

## 2. D1: [CLS] or mean of the tokens

### Categories

One logistic regression per category on standardized vectors. Two cells compare the pooling, and two more try a stronger regularization on the winner, because 768 dimensions for 4,301 sentences can overfit.

In [ ]:
def run_d1_category(pooling, C, retrain=False):
    def train_model():
        start = time.time()
        model = make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(C=C, max_iter=1000), n_jobs=-1))
        model.fit(train_features[pooling], train_labels)
        return {"model": model, "scores": model.predict_proba(val_features[pooling]), "fit_time_s": time.time() - start}

    run = cached_run(f"d1_category_{pooling}_C{C:g}", train_model, retrain)
    threshold, _ = best_global_threshold(run["scores"], val_labels)
    result = category_scores(val_labels, predict_labels(run["scores"], threshold))
    params = f"pooling={pooling}, C={C:g}"
    log_results([result_row("category", "D1 frozen arabert+lr", PREPROCESSING, params, "global", run["fit_time_s"], **result)])
    print(f"{params}: micro-F1 {result['micro_f1']:.4f}, macro-F1 {result['macro_f1']:.4f}, {run['fit_time_s']:.0f}s")
    return {"name": params, "pooling": pooling, **run, **result}

In [ ]:
d1_cls = run_d1_category("cls", 1.0)

In [ ]:
d1_mean = run_d1_category("mean", 1.0)

In [ ]:
best_pooling = "cls" if d1_cls["micro_f1"] > d1_mean["micro_f1"] else "mean"
d1_regularized = run_d1_category(best_pooling, 0.1)

In [ ]:
d1_strong = run_d1_category(best_pooling, 0.01)

In [ ]:
d1_category_runs = [d1_cls, d1_mean, d1_regularized, d1_strong]
plot_model_comparison(pd.DataFrame(d1_category_runs), "micro_f1", title="D1 categories on val")
best_d1_category = max(d1_category_runs, key=lambda run: run["micro_f1"])

### Polarity

The vector of the sentence plus a one-hot of the category, then a 3-class logistic regression with `class_weight="balanced"`.

In [ ]:
def pair_features(features, sentence_table, pair_table, pooling):
    row_of_sentence = dict(zip(sentence_table.sentence_id, range(len(sentence_table))))
    rows = pair_table.sentence_id.map(row_of_sentence).to_numpy()
    aspects = np.eye(len(classes))[pair_table.category.map(category_ids).to_numpy()]
    return np.hstack([features[pooling][rows], aspects])


def run_d1_polarity(pooling, C=1.0, retrain=False):
    def train_model():
        start = time.time()
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=1000))
        model.fit(pair_features(train_features, train, train_units, pooling), train_units.polarity.to_numpy(dtype=str))
        return {"model": model, "predicted": model.predict(pair_features(val_features, val, val_units, pooling)).astype(object),
                "fit_time_s": time.time() - start}

    run = cached_run(f"d1_polarity_{pooling}_C{C:g}", train_model, retrain)
    result = polarity_scores(val_units, run["predicted"])
    log_results([result_row("polarity", "D1 frozen arabert+lr", PREPROCESSING, f"pooling={pooling}, C={C:g}, balanced", fit_time_s=run["fit_time_s"], **result)])
    print(f"{pooling}, C={C:g}: macro-F1 {result['macro_f1']:.4f}, accuracy {result['accuracy']:.4f}, {run['fit_time_s']:.0f}s")
    return {"name": f"{pooling}, C={C:g}", "pooling": pooling, **run, **result}

In [ ]:
d1_polarity_cls = run_d1_polarity("cls")

In [ ]:
d1_polarity_mean = run_d1_polarity("mean")

In [ ]:
best_polarity_pooling = "cls" if d1_polarity_cls["macro_f1"] > d1_polarity_mean["macro_f1"] else "mean"
d1_polarity_regularized = run_d1_polarity(best_polarity_pooling, 0.1)

In [ ]:
d1_polarity_strong = run_d1_polarity(best_polarity_pooling, 0.01)

In [ ]:
d1_polarity_runs = [d1_polarity_cls, d1_polarity_mean, d1_polarity_regularized, d1_polarity_strong]
plot_model_comparison(pd.DataFrame(d1_polarity_runs), "macro_f1", title="D1 polarity on val")
final_polarity = max(d1_polarity_runs, key=lambda run: run["macro_f1"])

**Conclusions (Q8)**

_To fill in after the run: [CLS] against mean, and what frozen BERT gives against TF-IDF + LR._

## 3. D2: BiLSTM + attention on all the token states

The head of C2, with BERT states in place of word embeddings. The states are already computed, so a training only costs this small head.

In [ ]:
class Attention(nn.Module):
    def __init__(self, state_size):
        super().__init__()
        self.projection = nn.Linear(state_size, state_size)
        self.word_score = nn.Linear(state_size, 1, bias=False)

    def forward(self, states, mask):
        word_scores = self.word_score(torch.tanh(self.projection(states))).squeeze(2)
        weights = word_scores.masked_fill(~mask, -math.inf).softmax(dim=1)
        return (weights.unsqueeze(2) * states).sum(dim=1), weights


class BertStatesClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.dropout = nn.Dropout(DROPOUT)
        self.lstm = nn.LSTM(BERT_SIZE, HEAD_HIDDEN_SIZE, batch_first=True, bidirectional=True)
        self.attention = Attention(2 * HEAD_HIDDEN_SIZE)
        self.classifier = nn.Linear(2 * HEAD_HIDDEN_SIZE, len(classes))

    def encode(self, states, mask):
        packed = pack_padded_sequence(self.dropout(states), mask.sum(dim=1).cpu(), batch_first=True, enforce_sorted=False)
        packed_states, _ = self.lstm(packed)
        lstm_states, _ = pad_packed_sequence(packed_states, batch_first=True, total_length=states.size(1))
        return self.attention(lstm_states, mask)

    def forward(self, states, mask):
        summary, _ = self.encode(states, mask)
        return self.classifier(self.dropout(summary))

In [ ]:
def collate(items):
    states = [item["states"].float() for item in items]
    lengths = torch.tensor([len(row) for row in states])
    return {
        "states": pad_sequence(states, batch_first=True),
        "mask": torch.arange(lengths.max())[None, :] < lengths[:, None],
        "labels": torch.stack([item["labels"] for item in items]),
    }


def state_loader(features, labels, shuffle=False):
    label_rows = torch.tensor(labels, dtype=torch.float32)
    items = [{"states": states, "labels": label_row} for states, label_row in zip(features["tokens"], label_rows)]
    return DataLoader(items, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate)


def head_probabilities(model, loader):
    return collect_outputs(loader, lambda batch: model(batch["states"], batch["mask"])).sigmoid().numpy()


val_loader = state_loader(val_features, val_labels)


def train_d2(seed):
    set_seed(seed)
    model = BertStatesClassifier().to(DEVICE)
    loss_function = nn.BCEWithLogitsLoss()
    history = fit(
        model,
        state_loader(train_features, train_labels, shuffle=True),
        lambda model, batch: loss_function(model(batch["states"], batch["mask"]), batch["labels"]),
        lambda model: best_global_threshold(head_probabilities(model, val_loader), val_labels)[1].max(),
        torch.optim.AdamW(model.parameters(), lr=1e-3),
        epochs=MAX_EPOCHS, patience=PATIENCE,
    )
    return {"state": compact_state(model), "scores": head_probabilities(model, val_loader), "history": history}

In [ ]:
d2 = cached_run("v2_d2_category_seed42", lambda: train_d2(42))

d2_threshold, _ = best_global_threshold(d2["scores"], val_labels)
d2_result = category_scores(val_labels, predict_labels(d2["scores"], d2_threshold))
d2_fit_time = d2["history"][-1]["seconds"]
log_results([result_row("category", "D2 frozen arabert+bilstm attention", PREPROCESSING, "tokens", "global", d2_fit_time, **d2_result)])

plot_training_curves({"D2": d2["history"]}, "D2", "best micro-F1 on val")
plot_model_comparison(pd.DataFrame([
    {"name": "D1 logistic regression", "micro_f1": best_d1_category["micro_f1"]},
    {"name": "D2 BiLSTM + attention", "micro_f1": d2_result["micro_f1"]},
]), "micro_f1", title="Frozen BERT, categories on val")

In [ ]:
d2_seed_43 = cached_run("v2_d2_category_seed43", lambda: train_d2(43))

In [ ]:
d2_seed_44 = cached_run("v2_d2_category_seed44", lambda: train_d2(44))

**Conclusions (Q8)**

_To fill in after the run: does the head on all the tokens beat a single vector?_

## 4. Attention on 5 examples

Darker tokens get more attention. A long word can be cut into several tokens by the tokenizer.

In [ ]:
d2_model = BertStatesClassifier().to(DEVICE)
d2_model.load_state_dict(d2["state"])
d2_model.eval()

several_categories = [row for row, categories in enumerate(val.categories) if len(categories) > 1][:5]
shown_features = {"tokens": [val_features["tokens"][row] for row in several_categories]}
shown_batch = next(iter(state_loader(shown_features, val_labels[several_categories])))

with torch.inference_mode():
    _, shown_weights = d2_model.encode(shown_batch["states"].to(DEVICE), shown_batch["mask"].to(DEVICE))


def bert_tokens(text):
    return tokenizer.convert_ids_to_tokens(tokenizer(preprocess_text(text), truncation=True, max_length=MAX_LENGTH).input_ids)


show_attention(
    [bert_tokens(val.text.iloc[row]) for row in several_categories],
    [weights[mask].cpu().numpy() for weights, mask in zip(shown_weights, shown_batch["mask"].to(DEVICE))],
    [", ".join(val.categories.iloc[row]) for row in several_categories],
)

**Conclusions (Q7)**

_To fill in after the run: do the weights land on aspect and opinion words, and how do they compare with C2?_

## 5. Thresholds and full pipeline on val

In [ ]:
use_d2 = d2_result["micro_f1"] > best_d1_category["micro_f1"]
final_category_name = "D2" if use_d2 else "D1"
val_scores = d2["scores"] if use_d2 else best_d1_category["scores"]

global_threshold, f1_curve = best_global_threshold(val_scores, val_labels)
plot_f1_by_threshold(THRESHOLD_GRID, f1_curve, global_threshold, title=f"Micro-F1 by threshold ({final_category_name})")

holdout = compare_threshold_policies(val_scores, val_labels, val.review_id, seed=SEED)
plot_threshold_policies(holdout)

use_per_category = holdout.micro_f1_per_category.mean() > holdout.micro_f1_global.mean()
threshold_policy = "per_category" if use_per_category else "global"
chosen_thresholds = best_category_thresholds(val_scores, val_labels, global_threshold) if use_per_category else global_threshold

val_predictions = predict_labels(val_scores, chosen_thresholds)
plot_category_f1(category_report(val_labels, val_predictions, classes), title=f"F1 per category on val ({final_category_name}, {threshold_policy} threshold)")

In [ ]:
predicted_pairs = pairs_from_matrix(val, val_predictions, classes)
predicted_pairs["polarity"] = final_polarity["model"].predict(pair_features(val_features, val, predicted_pairs, final_polarity["pooling"]))

outcomes = pipeline_outcomes(val_pairs, predicted_pairs)
pipeline_result = pipeline_scores(outcomes)
results = log_results([result_row("pipeline", "frozen arabert", PREPROCESSING, final_category_name, threshold_policy, micro_f1=pipeline_result["pipeline_f1"])])

plot_pipeline_outcomes(outcomes)
print(pipeline_result)
plot_model_comparison(best_of_each_model(results, "category", "micro_f1"), "micro_f1", title="Categories on val")
plot_model_comparison(best_of_each_model(results, "polarity", "macro_f1"), "macro_f1", title="Polarity on val")

**Conclusions**

_To fill in after the run: frozen BERT against Part C and Part A. Part E compares it with fine-tuning._

## 6. Save the val scores

In [ ]:
thresholds_to_save = np.broadcast_to(chosen_thresholds, len(classes)).tolist()
save_scores("frozen_bert_category", "val", val_scores, {"sentence_id": val.sentence_id.to_numpy()}, classes,
            meta={"model": final_category_name, "threshold_policy": threshold_policy, "thresholds": thresholds_to_save})

## 7. Test predictions for Part F


Run this section only when the choices above are final. It writes the predictions on the test set for `06_final_test` and prints no score, so nothing here can influence a choice.

In [ ]:
test_opinions, test = load_test()
test_units = pair_units(pairs(test_opinions, test))
test_features = cached_run("v2_arabert_test_states", lambda: bert_states(test.text.tolist()))

if use_d2:
    test_loader = state_loader(test_features, category_matrix(test.categories, classes))
    test_score_list = []
    for run in [d2, d2_seed_43, d2_seed_44]:
        d2_model.load_state_dict(run["state"])
        test_score_list.append(head_probabilities(d2_model, test_loader))
    d2_model.load_state_dict(d2["state"])
else:
    test_score_list = [best_d1_category["model"].predict_proba(test_features[best_d1_category["pooling"]])]

test_scores = test_score_list[0]

In [ ]:
polarity_model = final_polarity["model"]
pooling = final_polarity["pooling"]
polarity_columns = [list(polarity_model.classes_).index(polarity) for polarity in POLARITIES]

test_polarity_scores = polarity_model.predict_proba(pair_features(test_features, test, test_units, pooling))[:, polarity_columns]
save_scores("frozen_bert_polarity", "test", [test_polarity_scores],
            {"sentence_id": test_units.sentence_id.to_numpy(), "category": test_units.category.to_numpy()}, POLARITIES)

test_pairs = pairs_from_matrix(test, predict_labels(test_scores, chosen_thresholds), classes)
test_pipeline_scores = polarity_model.predict_proba(pair_features(test_features, test, test_pairs, pooling))[:, polarity_columns]
save_scores("frozen_bert_pipeline", "test", [test_pipeline_scores],
            {"sentence_id": test_pairs.sentence_id.to_numpy(), "category": test_pairs.category.to_numpy()}, POLARITIES)

CPU latency of one sentence, through BERT and the category model:

In [ ]:
bert.cpu()
d2_model.cpu()


def predict_one_sentence(text):
    batch = tokenizer(preprocess_text(text), truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.inference_mode():
        states = bert(**batch).last_hidden_state
        if use_d2:
            return d2_model(states, batch["attention_mask"].bool())
        vectors = states[:, 0] if best_d1_category["pooling"] == "cls" else states.mean(dim=1)
        return best_d1_category["model"].predict_proba(vectors.numpy())


head_time = d2_fit_time if use_d2 else best_d1_category["fit_time_s"]
head_size = sum(parameter.numel() for parameter in d2_model.parameters()) if use_d2 else 769 * len(classes)

save_scores("frozen_bert_category", "test", test_score_list, {"sentence_id": test.sentence_id.to_numpy()}, classes, meta={
    "thresholds": thresholds_to_save,
    "train_s": head_time + train_features["seconds"],
    "polarity_train_s": final_polarity["fit_time_s"],
    "latency_ms": latency_ms(predict_one_sentence, test.text.head(50).tolist()),
    "size_mb": (sum(parameter.numel() for parameter in bert.parameters()) + head_size) * 4 / 1e6,
})